# Phase 3b — Finish Stage B in one session

Runs **B4** (sampler, two arms) and **B5** (eye-pair fusion) back to back, then prints
the whole Stage B table. Upload, attach the inputs, **Run All**.

Everything above B4 is already decided and fixed here: 512 px (B1), EfficientNet-B0
(B2), focal-ordinal head (B3). Only the sampler and fusion vary.

## Notebook settings (right-hand panel)

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Persistence | **Files only** — lets `--resume` pick up a killed run |
| Internet | **On** (git clone) |
| Environment | **Pin to original environment** |

## Inputs to add

`verify-dr-cache-512`, `verify-dr-idrid-masks`, and the Phase 2 manifests. Attach your
**earlier Stage B output** too if you have it — the comparison cell reads attached
results, so B1–B3 then appear in the same table.

## Cost

About **72 GPU-minutes**: roughly 18 + 18 + 36. Fusion is the slow one because it
encodes both eyes per sample.

## What to watch

- **`B4_natural`** is the run most likely to collapse. Natural prevalence is about 36:1
  toward grade 0, so check `n_pred` first — 1 or 2 means the run decides nothing,
  whatever its QWK.
- **`B4_balanced` is the last test of the prediction recorded in B2**: if it lifts
  grade-1 F1 to about 0.18, EfficientNet-B0 matches ResNet50's grade-1 behaviour at
  0.70x the cost and B2 is settled. If not, B2 reopens as a recorded deviation.

## 1 · Clone the repo

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"

# Print the commit actually in use. A stale checkout is the single most common
# cause of a confusing failure downstream: the notebook cell is new, the scripts
# on disk are not.
import subprocess
_sha = subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %s"],
                      capture_output=True, text=True).stdout.strip()
print("repo ready at", REPO_DIR)
print("checked out:", _sha)

## 2 · Check the GPU

In [ ]:
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. Set Accelerator to 'GPU T4 x2' in the right-hand panel, then "
        "re-run from the top.")

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory / 2**30:.1f} GB  sm_{p.major}{p.minor}")

## 3 · Helpers

In [ ]:
from pathlib import Path
from collections import Counter
import json, shlex, subprocess, sys, time, zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

def q(x):
    return shlex.quote(str(x))

def run(cmd):
    """Run a training job, streaming its output live.

    The earlier notebooks use capture_output=True, which is fine for a two-minute
    manifest build and useless here: you would see nothing at all until a
    three-hour job exited. Streaming means a per-epoch line appears as it happens,
    so a run that is going wrong can be stopped in epoch 1 rather than hour 3.
    """
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    code = proc.wait()
    if code != 0:
        if code == 2:
            print("\nExit 2 is an argument error. Usually the cloned scripts are stale:")
            print("re-run the clone cell at the top, then run from there.")
        raise RuntimeError(f"training failed with exit {code}")

def cache_roots():
    """Every directory that looks like a build_cache.py output root.

    build_cache.py writes <root>/<dataset>/cache_report.json, so a report file
    identifies its root two levels up. /kaggle/input is searched first: a stale
    copy in /kaggle/working must never silently win over the dataset you attached.
    """
    found = []
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        for root, _ in Counter(r.parent.parent for r in base.rglob("cache_report.json")).most_common():
            found.append(root)
    return found

def resolve_datasets(roots):
    """dataset -> the root holding the best copy of it.

    The cache is legitimately split across published datasets: the full build
    plus a later top-up. IDRiD's lesion masks ship as their own dataset
    ('verify-dr-idrid-masks'), so a single root shows idrid with zero mask
    channels even when the masks are attached. When a dataset appears in more
    than one root, the copy with more mask channels wins.
    """
    best = {}
    for root in roots:
        for d in sorted(x for x in root.iterdir() if x.is_dir()):
            if not (d / "cache_report.json").exists():
                continue
            masks = d / "masks"
            score = len(list(masks.iterdir())) if masks.is_dir() else 0
            if d.name not in best or score > best[d.name][1]:
                best[d.name] = (root, score)
    return {name: root for name, (root, _) in best.items()}

def extract_cache(dest=WORK / "cache512"):
    """Extract a cache published as a zip. Idempotent within a session.

    This costs GPU-session minutes, which come out of the 30 h/week quota. If you
    hit it every run, re-upload the cache to Kaggle as a *dataset* rather than as
    notebook output -- an uploaded zip is unpacked by Kaggle once, server-side.
    """
    for z in sorted(INPUT.rglob("*.zip")):
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
        except (zipfile.BadZipFile, OSError):
            continue
        if not any(n.endswith("cache_report.json") for n in names):
            continue
        marker = dest / ".extracted_from"
        if marker.exists() and marker.read_text().strip() == z.name:
            print(f"already extracted from {z.name}")
            return dest
        print(f"extracting {z.name} ({z.stat().st_size / 2**30:.1f} GB) -> {dest}")
        dest.mkdir(parents=True, exist_ok=True)
        started = time.time()
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        marker.write_text(z.name)
        print(f"extracted in {(time.time() - started) / 60:.1f} min")
        return dest
    return None

def find_manifest_dir():
    """Phase 2's output: the directory holding dataset_plan.json and the variants."""
    for base in (INPUT, WORK):
        if not base.exists():
            continue
        hits = sorted(base.rglob("dataset_plan.json"))
        if hits:
            return hits[0].parent
    return None

## 4 · Find the cache and the manifests

In [ ]:
ROOTS = cache_roots()
if not ROOTS and extract_cache():
    ROOTS = cache_roots()
MANIFEST_DIR = find_manifest_dir()

if not ROOTS:
    raise RuntimeError(
        "No cache found. Attach verify-dr-cache-512 (and verify-dr-idrid-masks) "
        "under Add Data in the right-hand panel.")
if MANIFEST_DIR is None:
    raise RuntimeError(
        "No manifests found. Attach the Phase 2 output (verify-dr-manifests), or "
        "add 02_manifests.ipynb as a notebook input.")

DATASETS = resolve_datasets(ROOTS)
print("cache roots  :")
for r in ROOTS:
    print("   ", r)
print("manifest dir :", MANIFEST_DIR)

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

print("\ndatasets in the cache:")
print(f"  {'dataset':<12}{'images':>8}  {'masks':>5}  root")
for name, root in sorted(DATASETS.items()):
    d = root / name
    images = d / 'images'
    # rglob, not glob: glob('*') counts direct children only, so a nested
    # layout reports 1 and looks like catastrophic data loss when nothing
    # is actually wrong.
    n_img = sum(1 for f in images.rglob('*') if f.suffix.lower() in IMAGE_SUFFIXES) \
        if images.is_dir() else 0
    n_msk = len([m for m in (d / 'masks').iterdir() if m.is_dir()]) \
        if (d / 'masks').is_dir() else 0
    print(f"  {name:<12}{n_img:>8}  {n_msk:>5}  {root}")

    if images.is_dir():
        subdirs = [x for x in images.glob('*') if x.is_dir()]
        if subdirs and n_img:
            print(f"               ^ nested under {len(subdirs)} subdirectorie(s), "
                  f"e.g. {subdirs[0].name}/ - fine, the manifest stores full paths")
    if n_img == 0:
        print(f"               ^ NO IMAGES - {name} is empty in every attached root")

print("\nmanifests available:")
for c in sorted(MANIFEST_DIR.glob("*.csv")):
    print("  ", c.name)

print("\nMasks are a Phase 4 concern. M1 grades whole images and reads images +")
print("grades only, so 0 mask channels here blocks nothing in Phase 3. What")
print("matters now is that the datasets your chosen manifest names have images.")

# Every root goes to --cache-root, so each dataset resolves to the root that
# actually holds it instead of all of them being forced onto one.
CACHE_FLAGS = ' '.join(q(r) for r in ROOTS)

## 5 · Run the batch

A per-epoch line appears as it happens. Each run checkpoints every epoch, and one
failure does not cost the others — the summary at the end says which succeeded.

In [ ]:
# ===================== EDIT THIS CELL, THEN RUN ALL =====================
# The three runs that close Stage B. Each writes its own results/<id>/.
#
# To reproduce all of Stage B from scratch instead, add the finished ones:
#   B1_res384/512/768, B2_resnet50, B3_softmax, B3_ordinal  (~2.5 GPU-h total)

COMMON = dict(
    manifest="eyepacs_balanced_1000.csv",
    image_size=512,                 # fixed by B1
    backbone="efficientnet_b0",     # fixed by B2
    head="ordinal_focal",           # fixed by B3
    epochs=10, lr=3e-4, epoch_samples=None,
)

BATCH = [
    dict(experiment="B4_natural",  sampler="natural",             batch_size=24),
    dict(experiment="B4_balanced", sampler="class_balanced",      batch_size=24),
    # Fusion encodes both eyes per sample: halve the batch, expect ~2x the time.
    # The sampler here is the incumbent, NOT a B4 winner -- if B4 picks something
    # else, re-run B5 with it (~36 min). Model selection stays a recorded decision,
    # so this notebook deliberately does not choose it for you mid-batch.
    dict(experiment="B5_fusion",   sampler="stratified_exposure", batch_size=12,
         eye_pair_fusion=True),
]
# =======================================================================

import traceback

outcomes = []
batch_started = time.time()

for i, spec in enumerate(BATCH, 1):
    cfg = {**COMMON, **spec}
    name = cfg.pop("experiment")
    print("=" * 72)
    print(f"[{i}/{len(BATCH)}]  {name}")
    print("=" * 72, flush=True)

    flags = [
        f"python {q(REPO_DIR / 'scripts/train_grading.py')}",
        f"--manifest {q(MANIFEST_DIR / cfg['manifest'])}",
        f"--experiment {q(name)}",
        f"--results-dir {q(RESULTS)}",
        f"--cache-root {CACHE_FLAGS}",
        f"--image-size {cfg['image_size']}",
        f"--backbone {cfg['backbone']}",
        f"--head {cfg['head']}",
        f"--sampler {cfg['sampler']}",
        f"--batch-size {cfg['batch_size']}",
        f"--epochs {cfg['epochs']}",
        f"--lr {cfg['lr']}",
        "--workers 2",
        "--resume",   # no-op when fresh; saves the batch if Kaggle kills the session
    ]
    if cfg.get("epoch_samples"):
        flags.append(f"--epoch-samples {cfg['epoch_samples']}")
    if cfg.get("eye_pair_fusion"):
        flags.append("--eye-pair-fusion")

    started = time.time()
    try:
        run(" ".join(flags))
        outcomes.append((name, f"ok       {(time.time() - started) / 60:5.1f} min"))
    except Exception as exc:
        # One bad run must not cost the other two.
        outcomes.append((name, f"FAILED   {exc}"))
        traceback.print_exc()
    print(flush=True)

print("=" * 72)
print(f"batch finished in {(time.time() - batch_started) / 60:.0f} min")
for name, outcome in outcomes:
    print(f"  {name:<16} {outcome}")

## 6 · The whole Stage B table

In [ ]:
import pandas as pd

def result_dirs():
    """Every directory holding <experiment>/metrics.json.

    Includes attached notebook outputs, so earlier Stage B runs appear in the
    same table when their dataset is attached as an input. Bounded to depth 2
    under /kaggle/input so it does not walk the 88k-image cache.
    """
    found = [RESULTS]
    if INPUT.exists():
        for base in INPUT.iterdir():
            if not base.is_dir():
                continue
            for cand in (base, base / "results"):
                if cand.is_dir() and any(cand.glob("*/metrics.json")):
                    found.append(cand)
    return found

rows, seen = [], set()
for d in result_dirs():
    for mp in sorted(d.glob("*/metrics.json")):
        m = json.loads(mp.read_text())
        b = m.get("best_val")
        if not b or m["experiment"] in seen:
            continue
        seen.add(m["experiment"])
        c = m["config"]
        rows.append({
            "experiment": m["experiment"],
            "res": c["image_size"],
            "backbone": c["backbone"].replace("efficientnet_", "eff_"),
            "head": c["head"], "sampler": c["sampler"][:10],
            "fusion": c["eye_pair_fusion"],
            "QWK": round(b["qwk"], 4), "macroF1": round(b["macro_f1"], 4),
            "g1_F1": round(b["per_class_f1"]["1"], 3),
            "g1_rec": round(b["per_class_recall"]["1"], 3),
            "n_pred": b.get("distinct_predictions"),
            "MAE": round(b["mae"], 3), "min": m["minutes"],
        })

if rows:
    table = pd.DataFrame(rows).sort_values("experiment")
    print(table.to_string(index=False))
    table.to_csv(WORK / "stage_b_summary.csv", index=False)

    print("\nn_pred of 1 means the run collapsed onto one grade; its QWK means nothing.")
    print("Noise floors from B1 (one seed per point): QWK 0.0276, macro-F1 0.0144,")
    print("grade-1 F1 0.0190, grade-1 recall 0.0220, MAE 0.0230. Judge each metric")
    print("against its own floor before calling a difference real.")
    print("\nCopy these into docs/04_experiment_register.md before the session ends.")
else:
    print("No completed runs found in", [str(d) for d in result_dirs()])

---
## 7 · Save

**Save Version → Save & Run All (Commit).** Publish `/kaggle/working/results` as
**`verify-dr-stage-b`** so later notebooks can read it.

Then copy the rows from § 6 into `docs/04_experiment_register.md` with the date, and
set each experiment's Status. `/kaggle/working` does not survive the session, and a run
whose number was never written down has to be paid for twice out of the same
30 GPU-h/week.

> Still validation only. No test set has been touched, and APTOS and Messidor-2 stay
> locked until the pre-registration is committed.